# 02 · Stage 1 — train the 24-layer Egyptian teacher

Fine-tunes Pocket TTS's English 24-layer teacher to Egyptian Arabic (new character text embedding, rest warm-started) — the official pocket-tts `finetune_language` recipe.

* Training runs as a **background process**: closing the browser does not stop it. Re-open this notebook and run the monitoring cells.
* **TensorBoard** shows losses, validation, learning rate, speed, GPU memory and **audio samples** of 5 fixed sentences every 1000 steps (EMA weights), including the diacritic-override pairs `المدينة` / `المدينةْ` and `العِلْم` / `العَلَم`.
* Stop with the STOP cell: it checkpoints the current step. Launching again resumes from the newest checkpoint.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

REPO = Path("/workspace/EgyPocket-TTS")          # where you cloned the repo
os.environ.setdefault("EGY_DATA", "/workspace/egypocket_data")
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")     # models + HF token persist on the volume
DATA = Path(os.environ["EGY_DATA"])
CATT_DIR = Path("/workspace/catt_tashkeel")      # the uploaded CATT folder
os.chdir(REPO)
sys.path.insert(0, str(REPO))
PY = sys.executable                              # the egypocket conda env
print("python:", PY)
print("repo  :", REPO)
print("data  :", DATA)

In [ ]:
from egypocket import notebook as nb
from egypocket.training.args import load_args
from egypocket.text.frontend import normalize_text

CONFIG = REPO / "configs/teacher_24l.yaml"
args = load_args(CONFIG)
RUN = args.run_dir
print(open(CONFIG, encoding="utf-8").read())
print("run dir:", RUN, "| effective batch:", args.batch_size * args.grad_accum_steps)

## TensorBoard

In [ ]:
print(nb.start_tensorboard(DATA / "runs"))

## Launch (or resume) training

In [ ]:
pid = nb.launch_training(CONFIG, RUN)
print("started, pid", pid)

## Monitor

The first steps include `torch.compile` warm-up (a few minutes). Then expect a step every ~1.5–2 s on a 4090 (effective batch 64). Run these cells whenever you want.

In [ ]:
print(nb.status(RUN, 30))

In [ ]:
nb.plot_scalars(RUN)

In [ ]:
sentences = [normalize_text(s) for s in args.sample_sentences]
nb.show_samples(RUN, sentences)

In [ ]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv

## When is the teacher good enough?

Listen to the samples. Typically the words become intelligible within a few thousand steps and the voice quality keeps improving afterwards. Signs of a good teacher: correct Egyptian pronunciation on sentences 0–1, `المدينة` vs `المدينةْ` audibly different (…-a vs …-at), `العِلْم` vs `العَلَم` different, `valid/flow_loss` flat. 30k steps is the default budget; stopping earlier is fine.

## Stop (graceful: checkpoint + exit)

In [ ]:
nb.stop_training(RUN)          # nb.stop_training(RUN, hard=True) kills immediately

In [ ]:
print(nb.status(RUN, 10))

Next: **03_distill_student.ipynb** (uses the newest teacher checkpoint).